Phishing 

In [3]:
import numpy as np 
import pandas  as pd
import joblib

from sklearn.model_selection import train_test_split,GridSearchCV,cross_val_score
from sklearn.preprocessing import LabelEncoder
from sklearn.feature_selection import RFE
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from xgboost import XGBClassifier

from sklearn.metrics import accuracy_score,classification_report
from imblearn.over_sampling import SMOTE


In [4]:
data=pd.read_csv("D:\Projects\Phishing\phishing.csv")

In [5]:
data.head()

,Index,UsingIP,LongURL,ShortURL,Symbol@,Redirecting//,PrefixSuffix-,SubDomains,HTTPS,DomainRegLen,...,UsingPopupWindow,IframeRedirection,AgeofDomain,DNSRecording,WebsiteTraffic,PageRank,GoogleIndex,LinksPointingToPage,StatsReport,class
0,0,1,1,1,1,1,-1,0,1,-1,...,1,1,-1,-1,0,-1,1,1,1,-1
1,1,1,0,1,1,1,-1,-1,-1,-1,...,1,1,1,-1,1,-1,1,0,-1,-1
2,2,1,0,1,1,1,-1,-1,-1,1,...,1,1,-1,-1,1,-1,1,-1,1,-1
3,3,1,0,-1,1,1,-1,1,1,-1,...,-1,1,-1,-1,0,-1,1,1,1,1
4,4,-1,0,-1,1,-1,-1,1,1,-1,...,1,1,1,1,1,-1,1,-1,-1,1


In [6]:
x=data.drop(columns=["class"])
y=data["class"]

print(x.shape)
print(y.shape)

(11054, 31)
(11054,)


In [7]:
# checking dataset is balanced are not
target_variable =y.value_counts()
print(target_variable)

class
 1    6157
-1    4897
Name: count, dtype: int64


In [8]:
# converting into ratio formate for balanceing the dataset 
ratio=target_variable.min()/target_variable.sum()

In [9]:
le=LabelEncoder()
y_encoder=le.fit_transform(y)

In [10]:
x_train,x_test,y_train,y_test=train_test_split(x,y_encoder,random_state=45,test_size=0.2,stratify=y_encoder)

In [11]:
# if the dataset was not balanced then using smoth method i will do the balance 
if ratio >4.5:
    smoth=SMOTE(random_state= 45)
    x_train_resampled,y_train_resampled=smoth.fit_resample(x_train,y_train)
    print(f"Original training shape: {x_train.shape}")
    print(f"Resampled training shape with SMOTE: {x_train_resampled.shape}")
    x_train,y_train = x_train_resampled,y_train_resampled
else:
    print("dataset was already balanced no need smothing ")




dataset was already balanced no need smothing 


In [12]:
# features selections(RFE)
rfe_engin=RandomForestClassifier(random_state=43)
selecter=RFE(estimator=rfe_engin,n_features_to_select=15,step=1)

selecter=selecter.fit(x_train,y_train)

x_train_selected=selecter.transform(x_train)
x_test_selected=selecter.transform(x_test)

selecter_columns=x.columns[selecter.support_]

print(f"Top Selected Features: {list(selecter_columns)}")

Top Selected Features: ['Index', 'UsingIP', 'PrefixSuffix-', 'SubDomains', 'HTTPS', 'DomainRegLen', 'RequestURL', 'AnchorURL', 'LinksInScriptTags', 'ServerFormHandler', 'AgeofDomain', 'DNSRecording', 'WebsiteTraffic', 'GoogleIndex', 'LinksPointingToPage']


In [13]:
# checking model 
models ={
    "RandomForestClassifier":RandomForestClassifier(random_state=42),
    "XGBClassifier":XGBClassifier(random_State=42),
    "Svc":SVC(random_state=99)

}

best_model=""
model_Score=0
for name ,model in models.items():
    model.fit(x_train_selected,y_train)
    pred=model.predict(x_test_selected)
    acc=accuracy_score(y_test,pred)
    print(f"accuracy_score:{acc:.4f}")

    if model_Score < acc:
        best_model=name
        model_Score=acc
print(f"\n>> Proceeding to tune top performer: {best_model}")
print(f"\n>> Proceeding to tune top performer: {model_Score}")

accuracy_score:0.9548
accuracy_score:0.9516


c:\Users\JANAKIRAM\AppData\Local\Programs\Python\Python311\Lib\site-packages\xgboost\training.py:200: UserWarning: [23:45:01] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:782: 
Parameters: { "random_State" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


accuracy_score:0.5572

>> Proceeding to tune top performer: RandomForestClassifier

>> Proceeding to tune top performer: 0.9547715965626413


In [14]:
if best_model=="xXGBClassifier":
    tuning_model=XGBClassifier(radndom_state=54,metric="logloss")
    param_grid={
        "n_estimator":[200,300],
        "max_depth":[10,20,None],
        "Learing_Rate":[0.01,0.1,0.2]}
else:
    tuning_model=RandomForestClassifier(random_state=33)
    param_grid={
        "n_estimators":[200,300],
        "max_depth":[10,20,None],
        "min_samples_split":[2,3]
    }

grid=GridSearchCV(estimator=tuning_model, param_grid=param_grid,cv=5,scoring="f1",n_jobs=1)
grid.fit(x_train_selected,y_train)

best_model=grid.best_estimator_
print("best_model_score",best_model)

cv_score=cross_val_score(best_model,x_train_selected,y_train,cv=5)
print(cv_score.mean())



best_model_score RandomForestClassifier(min_samples_split=3, n_estimators=200, random_state=33)
0.9578203934528544


In [15]:
final_prediction=best_model.predict(x_test_selected)
print(classification_report(y_test, final_prediction))

              precision    recall  f1-score   support

           0       0.96      0.94      0.95       979
           1       0.96      0.97      0.96      1232

    accuracy                           0.96      2211
   macro avg       0.96      0.96      0.96      2211
weighted avg       0.96      0.96      0.96      2211



In [16]:
print("\n--- Saving Model and Selector Artifacts ---")
joblib.dump(selecter, 'phishing_rfe_selector.pkl')
joblib.dump(best_model, 'final_phishing_model.pkl')
joblib.dump(le, 'target_label_encoder.pkl')

print("All pipeline components successfully exported!")


--- Saving Model and Selector Artifacts ---
All pipeline components successfully exported!


In [ ]:
print(pd.__version__)
print(sklearn.__version__)
print(xgboost.__version__)

2.3.2


NameError: name 'scikit' is not defined